# VLABench raw-policy baseline launcher (`select_fruit`)

Runs the bare pi0-fast VLA baseline (no planner, no executor) on VLABench `select_fruit`.

Baseline uses held-out Track-1 episode configs (`track_1_in_distribution`), first 10 episodes.

Launcher contract: `RUN_SCRIPT=examples/run_vlabench_raw_policy.py`

## 1 — Clone repo

Set `GITHUB_TOKEN` (repo-scope PAT) in the Colab **Secrets** sidebar before running.

In [ ]:
import os
from google.colab import userdata, drive

github_token = userdata.get('GITHUB_TOKEN')

if os.path.exists('/content/Longor'):
    os.chdir('/content/Longor')
    print('pulling repo...')
    !git pull
else:
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/Longor-Robotics.git"
    !git clone $CLONE_URL "/content/Longor"
    os.chdir('/content/Longor')

# mount Drive for VLABench assets tar
drive.mount('/content/drive')

## 2 — Headless rendering + app deps (VLAbench)

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

In [ ]:
%%bash
set -euo pipefail

cd /content/Longor
VLABENCH_PIN=cf588fe60c0c7282174fe979f5913170cfe69017
RRT_PIN=e51d95ee489a225220d6ae2a764c4111f6ba7d85

apt-get update
apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev python3.10-venv

# If .venv is broken: rm -rf .venv then re-run this cell.
if [ ! -d .venv ]; then
  python3.10 -m venv .venv || {
    python3.10 -m venv --without-pip .venv
    curl -fsSL https://bootstrap.pypa.io/get-pip.py | .venv/bin/python
  }
fi

source .venv/bin/activate

if [ ! -d .vendor/VLABench/.git ]; then
  git clone https://github.com/OpenMOSS/VLABench .vendor/VLABench
fi
git -C .vendor/VLABench checkout "$VLABENCH_PIN"

if [ ! -d .vendor/rrt-algorithms/.git ]; then
  git clone https://github.com/motion-planning/rrt-algorithms .vendor/rrt-algorithms
fi
git -C .vendor/rrt-algorithms checkout $RRT_PIN

python -m pip install -U pip
python -m pip install -r deps/requirements-vlabench.txt
touch .vendor/rrt-algorithms/rrt_algorithms/rrt/__init__.py
touch .vendor/rrt-algorithms/rrt_algorithms/search_space/__init__.py
touch .vendor/rrt-algorithms/rrt_algorithms/utilities/__init__.py
python -m pip install -e .vendor/rrt-algorithms
python -m pip install -e .vendor/VLABench
python -m pip install -e .

ASSETS_DIR=".vendor/VLABench/VLABench/assets"
ASSETS_TAR_DRIVE="/content/drive/MyDrive/Colab/assets.tar"
ASSETS_TAR_LOCAL="/content/assets.tar"

has_assets() {
  find "$ASSETS_DIR/obj/meshes/fruit" -name '*.xml' -print -quit 2>/dev/null | grep -q .
}

if has_assets; then
  echo "assets already present, skipping extract"
else
  test -f "$ASSETS_TAR_DRIVE" || {
    echo "missing $ASSETS_TAR_DRIVE" >&2
    echo "put VLABench assets.tar in Drive at MyDrive/Colab/assets.tar" >&2
    exit 1
  }

  echo "copying assets tar to local VM disk"
  cp "$ASSETS_TAR_DRIVE" "$ASSETS_TAR_LOCAL"

  echo "extracting assets from local VM tar"
  mkdir -p ".vendor/VLABench/VLABench"
  tar -xf "$ASSETS_TAR_LOCAL" -C ".vendor/VLABench/VLABench"
fi

# local copy is no longer needed after extraction
rm -f "$ASSETS_TAR_LOCAL"

python -c "import VLABench.robots, VLABench.tasks; from VLABench.envs import load_env; print('section 2 OK: VLABench importable')"

## 3 — Pi0-FAST policy venv (OpenPI)

In [ ]:
%%bash
set -euo pipefail

cd /content/Longor
VLA_VENV=/content/Longor/.vla-venv
VLA_PY="$VLA_VENV/bin/python"
OPENPI_DIR=/content/openpi

apt-get update
apt-get install -y python3.11 python3.11-venv python3.11-dev build-essential libevdev-dev pkg-config

# If .vla-venv is broken: rm -rf "$VLA_VENV" then re-run this cell.
if [ -x "$VLA_PY" ] && ! "$VLA_PY" -c 'import sys; raise SystemExit(0 if sys.version_info[:2]==(3,11) else 1)'; then
  rm -rf "$VLA_VENV"
fi

if [ ! -d "$VLA_VENV" ]; then
  python3.11 -m venv "$VLA_VENV" || {
    python3.11 -m venv --without-pip "$VLA_VENV"
    curl -fsSL https://bootstrap.pypa.io/get-pip.py | "$VLA_PY"
  }
fi

"$VLA_PY" -V

if [ ! -d "$OPENPI_DIR/.git" ]; then
  git clone https://github.com/Shiduo-zh/openpi "$OPENPI_DIR"
fi

"$VLA_PY" -m pip install -U pip setuptools wheel
"$VLA_PY" -m pip install -e "$OPENPI_DIR" \
  "lerobot @ git+https://github.com/huggingface/lerobot@0cf864870cf29f4738d3ade893e6fd13fbd7cdb5"
"$VLA_PY" -m pip install -r deps/requirements-vla-pi0fast-linux.txt
"$VLA_PY" -m pip install -e .

# pip cache is no longer needed after installs
"$VLA_PY" -m pip cache purge || true
rm -rf /root/.cache/pip

## 4 — Configure `.env` for baseline run

In [ ]:
from pathlib import Path

Path('.env').write_text(
    'LOG_LEVEL=DEBUG\n'
    'SAVE_VIDEO=1\n',
    encoding='utf-8',
)
print('.env written')

## 5 — Start VLA server + health

In [ ]:
%%bash
set -euo pipefail

cd /content/Longor
VLA_LOG=/tmp/longor_vlabench_vla_server.log
VLA_PYTHON=/content/Longor/.vla-venv/bin/python

if curl -fsS http://localhost:8765/health; then
  echo
  echo "VLA server already healthy"
  exit 0
fi

pkill -f "services/run_vla_server.py" || true
for _ in $(seq 1 60); do
  pgrep -f "services/run_vla_server.py" >/dev/null || break
  sleep 1
done
if pgrep -f "services/run_vla_server.py" >/dev/null; then
  echo "leftover VLA server still running; not starting another" >&2
  exit 1
fi

if ! "$VLA_PYTHON" -c "import config" 2>/dev/null; then
  echo "VLA venv missing longor install — re-run section 3 (or rm -rf .vla-venv first)" >&2
  exit 1
fi

nohup "$VLA_PYTHON" services/run_vla_server.py >"$VLA_LOG" 2>&1 &

sleep 2
if ! pgrep -f "services/run_vla_server.py" >/dev/null; then
  echo "VLA server exited immediately:" >&2
  tail -50 "$VLA_LOG" >&2
  exit 1
fi

elapsed=0
for _ in $(seq 1 180); do
  if curl -fsS http://localhost:8765/health; then
    echo
    echo "VLA server healthy"
    exit 0
  fi
  if ! pgrep -f "services/run_vla_server.py" >/dev/null; then
    echo "VLA server exited during startup:" >&2
    tail -50 "$VLA_LOG" >&2
    exit 1
  fi
  elapsed=$((elapsed + 5))
  echo "waiting for server to start... (${elapsed}s)"
  sleep 5
done

echo "VLA server not healthy after ${elapsed}s; see $VLA_LOG" >&2
tail -50 "$VLA_LOG" >&2
exit 1

## 6 — Run Track-1 raw-policy baseline (10 episodes)

Acceptance for this cost-bounded check: working only if `success_rate > 0.30` (at least 4/10).
This step writes a run log to `/tmp/longor_track1_baseline_10ep.log`.

In [ ]:
%%bash
set -uo pipefail

cd /content/Longor
RUN_LOG=/tmp/longor_track1_baseline_10ep.log

set +e
/content/Longor/.venv/bin/python examples/run_vlabench_raw_policy.py --task "select_fruit" --episodes 10 2>&1 | tee "$RUN_LOG"
BASELINE_EXIT=${PIPESTATUS[0]}
set -e

echo "BASELINE_EXIT_CODE=$BASELINE_EXIT"
grep -E "^PASS task=|^FAIL task=" "$RUN_LOG" | tail -n 1 || true

## 7 — Save run artifacts to Drive (before VM shutdown)

This exports the minimum data needed for post-run analysis and reproducibility.

In [ ]:
%%bash
set -euo pipefail

cd /content/Longor
STAMP=$(date +"%Y%m%d_%H%M%S")
OUT_DIR="/content/drive/MyDrive/Colab/longor_m5_track1_${STAMP}"
mkdir -p "$OUT_DIR"

cp /tmp/longor_track1_baseline_10ep.log "$OUT_DIR/" || true
cp /tmp/longor_vlabench_vla_server.log "$OUT_DIR/" || true
[ -f run.mp4 ] && cp run.mp4 "$OUT_DIR/" || true

git rev-parse HEAD > "$OUT_DIR/longor_git_sha.txt"
git -C .vendor/VLABench rev-parse HEAD > "$OUT_DIR/vlabench_git_sha.txt" || true
/content/Longor/.venv/bin/python -V > "$OUT_DIR/python_versions.txt"
/content/Longor/.vla-venv/bin/python -V >> "$OUT_DIR/python_versions.txt"

echo "Saved artifacts to: $OUT_DIR"
ls -lah "$OUT_DIR"

## 8 — Stop VLA server

In [ ]:
%%bash
set -euo pipefail
pkill -f "services/run_vla_server.py" || true
for _ in $(seq 1 30); do
  pgrep -f "services/run_vla_server.py" >/dev/null || exit 0
  sleep 1
done
echo "VLA server still running after shutdown" >&2
exit 1